In [19]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict

In [20]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int

    sr: float
    bpb: float
    boundary_percentage: float
    summary: str


In [21]:
def calculate_sr(state: BatsmanState) -> BatsmanState:
    sr = (state["runs"] / state["balls"]) * 100
    return { "sr": sr}


In [22]:
def calculate_bpb(state: BatsmanState) -> BatsmanState:
    bpb = state["balls"] / (state["fours"] + state["sixes"])
    return {"bpb": bpb}

In [23]:
def calculate_boundary_percentage(state: BatsmanState) -> BatsmanState:
    total_boundaries = state["fours"]*4 + state["sixes"]*6
    boundary_percentage = (total_boundaries / state["runs"]) * 100 if state["runs"] > 0 else 0
    return { "boundary_percentage": boundary_percentage}

In [24]:
def summary(state: BatsmanState) -> str:

    summary=f"""
    Strike rate: {state['sr']} \n
    Balls per boundary: {state['bpb']}\n
    Boundary percentage: {state['boundary_percentage']}\n
    """
    return { "summary": summary}

            

In [25]:
graph = StateGraph(BatsmanState)

In [26]:
graph.add_node("calculate_sr",calculate_sr)
graph.add_node("calculate_bpb",calculate_bpb)
graph.add_node("calculate_boundary_percentage",calculate_boundary_percentage)
graph.add_node("summary",summary)

graph.add_edge(START,"calculate_sr")
graph.add_edge(START,"calculate_bpb")
graph.add_edge(START,"calculate_boundary_percentage")
graph.add_edge("calculate_sr","summary")
graph.add_edge("calculate_bpb","summary")
graph.add_edge("calculate_boundary_percentage","summary")
graph.add_edge("summary",END)

workflow = graph.compile()


In [27]:
initial_state ={
    "runs": 120,
    "balls": 80,
    "fours": 10,
    "sixes": 5
}
   
    

final_state = workflow.invoke(initial_state)
print(final_state)


{'runs': 120, 'balls': 80, 'fours': 10, 'sixes': 5, 'sr': 150.0, 'bpb': 5.333333333333333, 'boundary_percentage': 58.333333333333336, 'summary': '\n    Strike rate: 150.0 \n\n    Balls per boundary: 5.333333333333333\n\n    Boundary percentage: 58.333333333333336\n\n    '}
